# Spectral elements: stiffness, convergence and mass lumping

This notebook compares conventional sixth-order Lagrange elements with GLL-based spectral elements on the same 2D mesh. It also demonstrates how GLL nodal quadrature produces a diagonal mass matrix.

**Diagnostic status:** The recorded sixth-order SEM run did not pass the positive-definiteness or convergence checks. Treat the performance and displacement comparisons as debugging output until these checks pass.

In [1]:
using LowLevelFEM, LinearAlgebra, SparseArrays
using IterativeSolvers, Preconditioners, IncompleteLU

[ Info: Precompiling LowLevelFEM [6171b9fb-adbf-4751-adb9-5faded75de07] (caches not reused: 2 for different dependency version already loaded, 4 for different Julia build configuration, 3 for file size changed)
Precompiling packages...
 155.0 s  ✓ LowLevelFEM
  1 dependency successfully precompiled in 156 seconds. 133 already precompiled.
[ Info: Precompiling LowLevelFEMIterativeSolversExt [55978d5f-b2ed-541f-a526-69ba3ceeb2d1] (caches not reused: 4 for different dependency version already loaded)
Precompiling packages...
   2.5 s  ✓ LowLevelFEMIterativeSolversExt (serial)
  1 dependency successfully precompiled in 4 seconds
[ Info: Precompiling LowLevelFEMPreconditionersExt [731aa7ef-afda-5ca4-bfd0-d21a0237399b] (caches not reused: 4 for different dependency version already loaded)
Precompiling packages...
   3.0 s  ✓ LowLevelFEMPreconditionersExt (serial)
  1 dependency successfully precompiled in 3 seconds
[ Info: Precompiling LowLevelFEMIncompleteLUExt [8123a700-b682-5da3-9778-c48f

### Mesh

Generate a structured mesh of sixth-order quadrilateral elements. The geometry and connectivity are shared by both formulations.

In [2]:
structured_rect_mesh(n=10, order=8)

### Standard Lagrange field

Define a 2D displacement field with the conventional (equidistant-node) Lagrange basis.

In [3]:
mat = Material("body", E=2e5, ν=0.3)
U = Field([mat], type=:VectorField, dim=2, fieldName=:u, basis=:lagrange);

In [4]:
ndofs(U)

13122

### Spectral field

Define the corresponding displacement field using the GLL-based spectral basis, without changing the mesh topology.

In [5]:
mat = Material("body", E=2e5, ν=0.3)
Us = Field([mat], type=:VectorField, dim=2, fieldName=:u, basis=:spectral);

### Weak-form operators

Use the same strain operator and constitutive law in both discretizations.

In [6]:
ε = SymGrad
D = ConstitutiveMatrix;

### Lagrange stiffness matrix

Assemble the plane-stress stiffness matrix with the conventional Lagrange basis.

In [7]:
K = ∫(ε(U) ⋅ D(:PlaneStress, mat) ⋅ ε(U));

### Spectral stiffness matrix

Assemble the same weak form with the spectral basis; only the interpolation changes.

In [8]:
Ks = ∫(ε(Us) ⋅ D(:PlaneStress, mat) ⋅ ε(Us));

### Spectral transformation diagnostics

Before interpreting the solver results, verify that the global GLL-to-Gmsh transformation is invertible and that both formulations assemble the same matrix in the original nodal representation.

In [9]:
sem = spectralTransformation(Us; details=true)
T, R = sem.T, sem.R
Isem = spdiagm(0 => ones(Float64, size(T, 2)))

println("Spectral transformation size: ", size(T))
println("Spectral DOFs: ", sem.spectral_dofs)
println("||R*T - I||∞: ", norm(R * T - Isem, Inf))
println("Relative stiffness difference before transformation: ",
        norm(K.A - Ks.A) / norm(K.A))

Spectral transformation size: (13122, 13122)
Spectral DOFs: 13122
||R*T - I||∞: 1.0519913868647706e-10
Relative stiffness difference before transformation: 0.0


### Boundary conditions and loading

Fix the left boundary and apply an identical horizontal traction on the right boundary.

In [10]:
bc = BoundaryCondition("left", ux=0, uy=0)

f = ∫(U ⋅ [1, 0], Γ="right")
fs = ∫(Us ⋅ [1, 0], Γ="right");

### Constrained-system diagnostics

The same essential boundary conditions must produce symmetric positive-definite reduced stiffness matrices. A failed Cholesky factorization or a failed CG solve is a correctness issue, not a performance result.

In [11]:
prepared_lagrange = LowLevelFEM.prepare_singlefield_system(Symmetric(K), f, [bc])
prepared_sem = LowLevelFEM.prepare_singlefield_system(Symmetric(Ks), fs, [bc])

println("Reduced sizes: Lagrange = ", size(prepared_lagrange.A),
        ", SEM = ", size(prepared_sem.A))
println("Lagrange is positive definite: ", isposdef(prepared_lagrange.A))
println("SEM is positive definite: ", isposdef(prepared_sem.A))

Reduced sizes: Lagrange = (12960, 12960), SEM = (12982, 12982)
Lagrange is positive definite: true
SEM is positive definite: true


### Direct solution: Lagrange

Solve the conventional system with a direct sparse solver and record the elapsed time.

In [12]:
@time u = solveField(Symmetric(K), f, support=[bc]);

  7.265401 seconds (10.29 M allocations: 892.590 MiB, 1.93% gc time, 97.14% compilation time)


### Direct solution: SEM

Solve the spectral system under the same boundary conditions.

In [13]:
@time us = solveField(Symmetric(Ks), fs, support=[bc]);

  0.790529 seconds (10.78 k allocations: 3.204 GiB, 30.95% gc time, 0.89% compilation time)


### Direct-solver agreement

Compare the reconstructed displacement vectors. LowLevelFEM returns SEM results in the original Gmsh nodal storage basis, so the DoF vectors should agree to solver accuracy.

In [14]:
err = norm(DoFs(u) - DoFs(us)) / norm(DoFs(u))

0.0008142390567358233

In [15]:
pl = LowLevelFEM.prepare_singlefield_system(
    Symmetric(K), f, [bc]
)

ps = LowLevelFEM.prepare_singlefield_system(
    Symmetric(Ks), fs, [bc]
)

println("Lagrange:")
@show length(pl.fixed_r)
@show length(pl.free_r)

println("\nSEM:")
@show length(ps.fixed_r)
@show length(ps.free_r)

sem = spectralTransformation(Us; details=true, check=true)
@show sem.inverse_error

@show norm(K[:,:] - Ks[:,:]) / norm(K[:,:])


Lagrange:
length(pl.fixed_r) = 162
length(pl.free_r) = 12960

SEM:
length(ps.fixed_r) = 140
length(ps.free_r) = 12982
sem.inverse_error = 1.0519913868647706e-10
norm(K[:, :] - Ks[:, :]) / norm(K[:, :]) = 0.0


0.0

### Iterative solution: Lagrange

Solve the symmetric stiffness system using preconditioned conjugate gradients (CG).

In [16]:
@time u2 = solveField(Symmetric(K), f, support=[bc], solver=:cg, preconditioner=:none, solveroptions=(reltol=1e-9, maxiter=20000));

 35.186141 seconds (440.30 k allocations: 159.628 MiB, 0.19% gc time, 1.74% compilation time)


### Iterative solution: SEM

Repeat the same iterative solve for the spectral discretization.

In [17]:
@time us2 = solveField(Symmetric(Ks), fs, support=[bc], solver=:cg, preconditioner=:none, solveroptions=(reltol=1e-9, maxiter=20000));

  4.888920 seconds (13.54 k allocations: 3.026 GiB, 5.46% gc time)


### CG iteration counts

`solveField` currently discards the convergence history provided by `IterativeSolvers.jl`. As a temporary diagnostic, reproduce its constrained systems and incomplete-Cholesky preconditioner, and rerun CG with `log=true` to report the actual iteration counts. **This additional cell runs both CG solves again** and is not included in the timings above.

In [18]:
# Temporary diagnostic; uses LowLevelFEM's current internal solver helpers.
function cg_iteration_count(K, rhs, support)
    prepared = LowLevelFEM.prepare_singlefield_system(K, rhs, support)
    P = LowLevelFEM._build_preconditioner(Val(:ilu), prepared.A)
    _, history = cg(prepared.A, vec(prepared.B[:, 1]); Pl=P, log=true)
    return history.iters, history.isconverged
end

iters_lagrange, converged_lagrange = cg_iteration_count(Symmetric(K), f, [bc])
iters_sem, converged_sem = cg_iteration_count(Symmetric(Ks), fs, [bc])

println("Lagrange CG: $iters_lagrange iterations (converged: $converged_lagrange)")
println("SEM CG:      $iters_sem iterations (converged: $converged_sem)")

Lagrange CG: 2 iterations (converged: true)
SEM CG:      2 iterations (converged: true)


### Iterative-solver agreement

Check the relative difference between the two CG solutions.

In [19]:
err = norm(DoFs(u2) - DoFs(us2)) / norm(DoFs(u2))

0.000814239057693568

### Consistent mass matrix: Lagrange

Assemble the standard displacement mass matrix. Its off-diagonal entries are generally nonzero.

In [20]:
M = ∫(U ⋅ U)
M[:,:]

13122×13122 SparseMatrixCSC{Float64, Int64} with 1283202 stored entries:
⎡⡿⣯⣿⣁⣀⣨⡍⠀⠀⣤⡍⠀⠀⣼⠁⠀⢠⣼⠃⠀⢠⡟⠀⠀⠤⠟⠀⠀⠼⠃⠀⠠⠼⠓⠶⠶⠟⠶⠶⠿⎤
⎢⠟⢻⡿⣯⣍⠉⠿⣭⠉⠉⠉⠉⠉⠉⠉⠉⠉⠉⠉⠉⠉⠛⠛⠛⠛⠛⠛⠛⠛⠒⠒⠒⠒⠒⠒⠒⠒⠒⠒⠒⎥
⎢⡀⣸⡇⠙⢻⣶⡄⠘⠳⣆⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠃⠉⡟⣧⣀⠉⢿⣷⣀⠉⢻⣄⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⣤⡇⠀⠹⢦⡄⠘⠿⣧⡄⠈⠷⣤⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠃⠉⡇⠀⠀⠀⠛⢶⡀⠉⢻⣶⡄⠘⠳⣆⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⣀⣤⡇⠀⠀⠀⠀⠈⠙⣧⣀⠉⢿⣷⣀⠉⢻⣄⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠁⠀⡇⠀⠀⠀⠀⠀⠀⠀⠹⢦⡄⠘⠿⣧⣄⠈⠷⣤⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⣀⣶⡇⠀⠀⠀⠀⠀⠀⠀⠀⠀⠛⢶⡀⠙⢻⣶⡀⠘⠳⣆⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠉⠀⡇⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠙⣧⣀⠈⢿⣷⣀⠉⢻⣄⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⣤⠶⣧⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠹⢦⡄⠘⠿⣧⣄⠈⠷⣤⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⣿⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠛⢶⡀⠙⢻⣶⡄⠘⠳⣆⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⣤⠇⣿⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠙⣧⣀⠉⠿⣧⣀⠉⢻⣄⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⣿⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠹⢦⡄⠘⠿⣧⣄⠈⠷⣤⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠶⠃⢻⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠛⢶⡀⠙⢻⣶⡄⠘⠳⣆⠀⠀⠀⠀⠀⠀⎥
⎢⠀⡀⢸⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠙⣧⣀⠉⢿⣷⣀⠉⢻⣄⡀⠀⠀⠀⎥
⎢⢶⠃⢸⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠹⢦⡄⠘⠻⣦⣄⠈⠷⣄⠀⠀⎥
⎢⢸⡇⢸⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠛⢶⡀⠙⢻⣶⡄⠙⢷⡄⎥
⎢⢻⡅⢸⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠙⢧⣄⠉⠿⣧⡄⠉⎥
⎣⣼⡇⢸⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠙⠷⡄⠉⠿⣧⎦

### Consistent mass matrix: SEM

Using conventional quadrature, the spectral element also has a non-diagonal consistent mass matrix.

In [21]:
Ms1 = ∫(Us ⋅ Us)
Ms1[:,:]

13122×13122 SparseMatrixCSC{Float64, Int64} with 1283202 stored entries:
⎡⡿⣯⣿⣁⣀⣨⡍⠀⠀⣤⡍⠀⠀⣼⠁⠀⢠⣼⠃⠀⢠⡟⠀⠀⠤⠟⠀⠀⠼⠃⠀⠠⠼⠓⠶⠶⠟⠶⠶⠿⎤
⎢⠟⢻⡿⣯⣍⠉⠿⣭⠉⠉⠉⠉⠉⠉⠉⠉⠉⠉⠉⠉⠉⠛⠛⠛⠛⠛⠛⠛⠛⠒⠒⠒⠒⠒⠒⠒⠒⠒⠒⠒⎥
⎢⡀⣸⡇⠙⢻⣶⡄⠘⠳⣆⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠃⠉⡟⣧⣀⠉⢿⣷⣀⠉⢻⣄⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⣤⡇⠀⠹⢦⡄⠘⠿⣧⡄⠈⠷⣤⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠃⠉⡇⠀⠀⠀⠛⢶⡀⠉⢻⣶⡄⠘⠳⣆⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⣀⣤⡇⠀⠀⠀⠀⠈⠙⣧⣀⠉⢿⣷⣀⠉⢻⣄⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠁⠀⡇⠀⠀⠀⠀⠀⠀⠀⠹⢦⡄⠘⠿⣧⣄⠈⠷⣤⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⣀⣶⡇⠀⠀⠀⠀⠀⠀⠀⠀⠀⠛⢶⡀⠙⢻⣶⡀⠘⠳⣆⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠉⠀⡇⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠙⣧⣀⠈⢿⣷⣀⠉⢻⣄⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⣤⠶⣧⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠹⢦⡄⠘⠿⣧⣄⠈⠷⣤⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⣿⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠛⢶⡀⠙⢻⣶⡄⠘⠳⣆⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⣤⠇⣿⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠙⣧⣀⠉⠿⣧⣀⠉⢻⣄⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⣿⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠹⢦⡄⠘⠿⣧⣄⠈⠷⣤⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠶⠃⢻⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠛⢶⡀⠙⢻⣶⡄⠘⠳⣆⠀⠀⠀⠀⠀⠀⎥
⎢⠀⡀⢸⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠙⣧⣀⠉⢿⣷⣀⠉⢻⣄⡀⠀⠀⠀⎥
⎢⢶⠃⢸⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠹⢦⡄⠘⠻⣦⣄⠈⠷⣄⠀⠀⎥
⎢⢸⡇⢸⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠛⢶⡀⠙⢻⣶⡄⠙⢷⡄⎥
⎢⢻⡅⢸⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠙⢧⣄⠉⠿⣧⡄⠉⎥
⎣⣼⡇⢸⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠙⠷⡄⠉⠿⣧⎦

### GLL nodal quadrature: diagonal mass matrix

GLL nodal quadrature gives a diagonal mass matrix **in spectral coordinates**. LowLevelFEM returns its matrix in Gmsh storage coordinates; the assembled matrix shown below is therefore generally not diagonal. We verify diagonality after transforming back to the compact spectral space.

In [22]:
Ms2 = ∫(Us ⋅ Us, gauss=:gll)
Ms2[:,:]

13122×13122 SparseMatrixCSC{Float64, Int64} with 1283202 stored entries:
⎡⡿⣯⣿⣁⣀⣨⡍⠀⠀⣤⡍⠀⠀⣼⠁⠀⢠⣼⠃⠀⢠⡟⠀⠀⠤⠟⠀⠀⠼⠃⠀⠠⠼⠓⠶⠶⠟⠶⠶⠿⎤
⎢⠟⢻⡿⣯⣍⠉⠿⣭⠉⠉⠉⠉⠉⠉⠉⠉⠉⠉⠉⠉⠉⠛⠛⠛⠛⠛⠛⠛⠛⠒⠒⠒⠒⠒⠒⠒⠒⠒⠒⠒⎥
⎢⡀⣸⡇⠙⢻⣶⡄⠘⠳⣆⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠃⠉⡟⣧⣀⠉⢿⣷⣀⠉⢻⣄⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⣤⡇⠀⠹⢦⡄⠘⠿⣧⡄⠈⠷⣤⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠃⠉⡇⠀⠀⠀⠛⢶⡀⠉⢻⣶⡄⠘⠳⣆⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⣀⣤⡇⠀⠀⠀⠀⠈⠙⣧⣀⠉⢿⣷⣀⠉⢻⣄⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠁⠀⡇⠀⠀⠀⠀⠀⠀⠀⠹⢦⡄⠘⠿⣧⣄⠈⠷⣤⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⣀⣶⡇⠀⠀⠀⠀⠀⠀⠀⠀⠀⠛⢶⡀⠙⢻⣶⡀⠘⠳⣆⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠉⠀⡇⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠙⣧⣀⠈⢿⣷⣀⠉⢻⣄⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⣤⠶⣧⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠹⢦⡄⠘⠿⣧⣄⠈⠷⣤⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⣿⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠛⢶⡀⠙⢻⣶⡄⠘⠳⣆⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⣤⠇⣿⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠙⣧⣀⠉⠿⣧⣀⠉⢻⣄⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⣿⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠹⢦⡄⠘⠿⣧⣄⠈⠷⣤⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠶⠃⢻⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠛⢶⡀⠙⢻⣶⡄⠘⠳⣆⠀⠀⠀⠀⠀⠀⎥
⎢⠀⡀⢸⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠙⣧⣀⠉⢿⣷⣀⠉⢻⣄⡀⠀⠀⠀⎥
⎢⢶⠃⢸⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠹⢦⡄⠘⠻⣦⣄⠈⠷⣄⠀⠀⎥
⎢⢸⡇⢸⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠛⢶⡀⠙⢻⣶⡄⠙⢷⡄⎥
⎢⢻⡅⢸⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠙⢧⣄⠉⠿⣧⡄⠉⎥
⎣⣼⡇⢸⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠙⠷⡄⠉⠿⣧⎦

### Verify diagonality in spectral coordinates

Because the public `SystemMatrix` uses the original Gmsh nodal representation, test the transformed matrix $T^T M_{\mathrm{GLL}} T$. The off-diagonal norm should be close to roundoff if the transformation and quadrature are consistent.

In [23]:
M_sem = sparse(T' * Ms2.A * T)
D_sem = spdiagm(0 => diag(M_sem))
relative_offdiag = norm(M_sem - D_sem) / norm(M_sem)

println("Relative off-diagonal norm of the spectral mass matrix: ", relative_offdiag)
println("Minimum diagonal entry: ", minimum(diag(M_sem)))
println("Number of explicitly stored entries in public Gmsh matrix: ", nnz(Ms2.A))
println("Number of explicitly stored entries in spectral matrix: ", nnz(M_sem))

Relative off-diagonal norm of the spectral mass matrix: 1.141046584381886e-11
Minimum diagonal entry: 1.9290123402811797e-6
Number of explicitly stored entries in public Gmsh matrix: 1283202
Number of explicitly stored entries in spectral matrix: 1283202
